In [ ]:
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
import re
from typing import Dict

In [47]:
class JargonExpanderTool:
    """
    Tool to expand internal jargons into their full descriptive text
    within an email or message.
    """

    def __init__(self):
        self.jargon_map: Dict[str, str] = {
            "POV": "point of view",
            "TAS": "technical architecture stack",
            "WIP": "in progress",
            "POC": "proof of concept",
            "MVP": "minimum viable product",
            "UAT": "user acceptance testing",
            "SLA": "service level agreement",
            "PING": "reach out",
            "POV": "point of view",
            "FYI": "for your information",
            "ASAP": "as soon as possible",
            "EOD": "end of day",
            "ETA": "estimated time of arrival",
            "RACI": "responsible, accountable, consulted, informed",
            "TBD": "to be decided",
            "TBA": "to be announced",
            "OOO": "out of office",
            "BRB": "be right back",
            "IMHO": "in my humble opinion",
            "TLDR": "too long; didn’t read",
            "THX": "thanks",
            "NP": "no problem"
        }

    async def expand_jargons(self, email_text: str) -> str:
        """
        Replaces known internal jargons in the given email text
        with their full descriptive meaning.

        Args:
            email_text (str): Raw email content containing jargons

        Returns:
            str: Email text with all known jargons expanded
        """

        expanded_text = email_text

        for jargon, full_text in self.jargon_map.items():
            pattern = rf"\b{re.escape(jargon)}\b"
            expanded_text = re.sub(pattern, full_text, expanded_text)

        return expanded_text

In [54]:
async def main(email_content:str):
    jargon_tool = JargonExpanderTool()

    async with (
        AzureCliCredential() as credential,
        ChatAgent(
            chat_client=AzureAIAgentClient(async_credential=credential),
            name="EmailClarityAgent",
            instructions="""
            You receive internal emails.
            If the content contains corporate or project jargons, always use the jargon expansion tool to clarify them
            before responding. 
            """,
            tools=[jargon_tool.expand_jargons],
            store = True #just to remove the conversation_id not set warning 😊
        ) as agent,
    ): 

        result = await agent.run(email_content)
        print(result.text)

In [55]:
email = """
        Hi Team,

        FYI, sharing my POV on the current MVP status.

        The POC is currently in WIP and progressing as planned. From a TAS perspective, the overall design looks stable, and we are targeting 
        UAT by EOD tomorrow. The ETA for completing all validation activities is ASAP, subject to meeting the agreed SLA.

        IMHO, we should finalize the RACI before moving ahead, as a few ownership items are still TBD. Some dependencies are also TBA based on 
        feedback from the business stakeholders.

        Please PING the relevant owners and BRB with updates once you have clarity. If anyone is OOO today, NP — we can circle back once 
        they are available.

        MVP is on track, minor action items pending, and next steps are clear.

        THX everyone for the continued support.

        Best regards,
        Atanu
        """

expanded_email = await main(email)
print(expanded_email)

Hi Team,

For your information, I'm sharing my point of view on the current minimum viable product (MVP) status.

The proof of concept (POC) is currently in progress and advancing as planned. From a technical architecture stack (TAS) perspective, the overall design looks stable, and we are targeting user acceptance testing (UAT) by the end of the day (EOD) tomorrow. The estimated time of arrival (ETA) for completing all validation activities is as soon as possible (ASAP), subject to meeting the agreed service level agreement (SLA).

In my humble opinion (IMHO), we should finalize the responsible, accountable, consulted, informed (RACI) chart before moving ahead, as a few ownership items are still to be decided (TBD). Some dependencies are also to be announced (TBA) based on feedback from the business stakeholders.

Please reach out to the relevant owners and be right back (BRB) with updates once you have clarity. If anyone is out of office (OOO) today, no problem — we can circle back o